<h2><font color="#004D7F" size=5>Módulo 4: Stacking</font></h2>


<h1><font color="#004D7F" size=6> 5. Blending</font></h1>
<br><br>
<div style="text-align: right">
<font color="#004D7F" size=3>Manuel Castillo-Cara</font><br>
<font color="#004D7F" size=3>Aprendizaje Automático II</font><br>
<font color="#004D7F" size=3>Universidad Nacional de Educación a Distancia</font>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web personal](https://www.manuelcastillo.eu/udemy/)

---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>


* [1. Algoritmo Blending](#section1)
* [2. Blending para Clasificación](#section3)
    * [2.1. Por etiqueta de clases](#section31)
    * [2.2. Por probabilidades](#section32)
    * [2.3. Modelos contribuyentes](#section32)
* [3. Blending para Clasificación](#section3)
    * [3.1. Resultado Blending](#section31)
    * [3.2. Modelos contribuyentes](#section32)
* [Ejercicios](#sectionEj)

---

<a id="section0"></a>
# <font color="#004D7F">0. Contexto</font>

Blending, en lugar de ajustar el metamodelo a las predicciones realizadas por el modelo base, se ajusta a las predicciones realizadas en un conjunto de datos reservado. En este tutorial, trabajaremos:
- Blending son un tipo de Stacking en el que el metamodelo se ajusta mediante predicciones en un conjunto de datos de validación reservados en lugar de predicciones fuera del fold.
- Cómo desarrollarBlending, incluidas funciones para entrenar el modelo y hacer predicciones sobre nuevos datos.
- Cómo evaluar Blending para problemas de modelado predictivo de clasificación y regresión.

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section1"></a>
# <font color="#004D7F"> 1. Algoritmo Blending</font>

- Mantine la arquitectura de dos niveles:
    - __Modelos de nivel 0 (Modelo base)__: los modelos se ajustan a los datos de entrenamiento y cuyas predicciones se comp
    - __Modelo de nivel 1 ( Metamodelo)__: modelo que aprende cómo combinar mejor las predicciones de los modelos.

La diferencia entre Stacking y Blending:
- __Blending__: el metamodelo se entrena en base a predicciones realizadas en un conjunto de datos reservado (e.g., validación).
- __Stacking__: el metamodelo se entrena en predicciones fuera de fold realizadas durante la validación cruzada de k veces.
  
<figure><center>
  <img src="data/Stacked.png" width="450" height="350" alt="Gráfica">
  <figcaption><blockquote>Gradient Boosting Machine. Extraída de <a href="http://dx.doi.org/10.5220/0006431602160222">Demand Prediction Using Machine Learning Methods and Stacked Generalization</a></blockquote></figcaption>
</center></figure>

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section2"></a> 
# <font color="#004D7F"> 2. Blending para Clasificación</font>

Veamos un ejemplo para un problema de clasificación

<a id="section21"></a> 
## <font color="#004D7F"> 2.1. Por etiqueta de clases</font>

Lo primero que vamos a ver el qué obtenemos de resultado en cada modelo de nivel 0.

Usamos la función `make_classification()` para crear un problema de clasificación binaria sintética con 1000 ejemplos y 20 características de entrada.

In [ ]:
from sklearn.datasets import make_classification

def get_dataset():
	X, y = make_classification(n_samples=5000, n_features=20, n_informative=10, n_redundant=10, random_state=1)
	return X, y

- Scikit-learn no admite de forma nativa Blending.
- Tenemos que crar una función que tenga los modelos a desarrollar: LoR, k-NN, CART, SVM y GNB.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
</div>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB


def get_models():
	models = ???
	models.append(???)
	models.append(???)
	models.append(???)
	models.append(???)
	models.append(???)
	return models

- Los modelos base se ajustan a un conjunto de datos de entrenamiento.
- El metamodelo se ajusta a las predicciones realizadas por cada modelo base en un conjunto de datos reservado.
- Primero, enumeramos y entrenamos la lista de modelos base en train. También en este ciclo, hacemos la predicción en el conjunto de datos de reserva (validación) y almacenar las predicciones.
- Unimos esto en una función llamada `fit_ensemble()` que entrena el modelo Blending en train y validación.


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre [`hstack`](https://numpy.org/doc/stable/reference/generated/numpy.hstack.html)
</div>

In [ ]:
from sklearn.linear_model import LogisticRegression
from numpy import hstack

def fit_ensemble(models, X_train, X_val, y_train, y_val):
	# Entrenamos los modelos nivel 0 en train y nivel 1 en validación
	meta_X = list()
	???:
		???
		yhat = ???
		# redimensionamos las predicciones en una matriz de una columna
		yhat = ???
		# almacenamos las predicciones como entrada para Blending
		???
	# creamos un array 2D de las predicciones, cada modelo como una característica
	meta_X = ???
	# definimos el modelo de nivel 1
	blender = ???
	# ajustamos las predicciones en validación
	???
	return blender

El siguiente paso es utilizar Blending para hacer predicciones sobre nuevos datos. Este es un proceso de dos pasos. 
1. El primer paso es utilizar cada modelo base para hacer una predicción.
2. Luego, estas predicciones se reúnen y se utilizan como entrada para el metamodelo para realizar la predicción final.

La función `predict_ensemble()` implementa la lista de modelos nivel 0, Blending y los dataset test. 
Devolverá un conjunto de predicciones para el conjunto de datos.

In [ ]:
def predict_ensemble(models, blender, X_test):
	# hacer prediciones con los modelos nivel 0
	meta_X = list()
	???:
		# predict de los nivel 0
		yhat = ???
		# redimensiona las predicciones en una matriz con una columna
		yhat = ???
		# almacenar las predicciones
		???
	# Crear un array 2D de las predicciones con cada modelo como entrada
	meta_X = ???
	# devolvemos el predict
	return ???

Dividimos el conjunto de datos en train y test (50-50). El subconjunto train lo volvemos a dividir (67-33) para entrenar los de nivel 0 y el otro al nivel 1. 

In [ ]:
from sklearn.model_selection import train_test_split

X, y = ???
X_train_full, X_test, y_train_full, y_test = ???
X_train, X_val, y_train, y_val = ???
print('Train: %s, Val: %s, Test: %s' % (X_train.shape, X_val.shape, X_test.shape))

Evaluamos en test de nuestro modelo ensemble.

In [ ]:
from sklearn.metrics import accuracy_score

models = ???
# entrenamiento ensemble
blender = ???
# Predicciones en test
yhat = ???
# Evaluar predicciones
score = ???
print('Accuracy Blending: %.3f' % (score*100))

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

<a id="section22"></a> 
## <font color="#004D7F"> 2.2. Por probabilidades</font>

Una alternativa es hacer que cada modelo prediga las probabilidades de clase y utilizar el metamodelo para combinar las probabilidades (Voting _soft_). 

Primero, debemos configurar los modelos para que devuelvan probabilidades, como el modelo SVM.

In [ ]:
# get a list of base models
def get_models():
	models = list()
	models.append(('lr', LogisticRegression()))
	models.append(('knn', KNeighborsClassifier()))
	models.append(('cart', DecisionTreeClassifier()))
	models.append(???)
	models.append(('bayes', GaussianNB()))
	return models

- Debemos cambiar los modelos base para predecir probabilidades en lugar de etiquetas de clase nítidas. 
- Esto se puede lograr llamando a la función `predict_proba()` en la función `fit_ensemble()`.

In [ ]:
def fit_ensemble(models, X_train, X_val, y_train, y_val):
	meta_X = list()
	for _, model in models:
		model.fit(X_train, y_train)
		yhat = ???
		meta_X.append(yhat)
	meta_X = hstack(meta_X)
	blender = LogisticRegression()
	blender.fit(meta_X, y_val)
	return blender

También necesitamos cambiar las predicciones realizadas por los modelos base cuando utilizamos el modelo combinado para hacer predicciones sobre datos nuevos.

In [ ]:
def predict_ensemble(models, blender, X_test):
	meta_X = list()
	for _, model in models:
		yhat = ???
		meta_X.append(yhat)
	meta_X = hstack(meta_X)
	return blender.predict(meta_X)

Veamos los resultados.

In [ ]:
X, y = get_dataset()
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.33, random_state=1)
print('Train: %s, Val: %s, Test: %s' % (X_train.shape, X_val.shape, X_test.shape))
models = get_models()
blender = fit_ensemble(models, X_train, X_val, y_train, y_val)
yhat = predict_ensemble(models, blender, X_test)
score = accuracy_score(y_test, yhat)
print('Accuracy Blending: %.3f' % (score*100))

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

<a id="section23"></a> 
## <font color="#004D7F"> 2.3. Modelos contribuyentes</font>

Cada modelo base puede ajustarse a todo train y evaluarse en test (al igual que Blending).

In [ ]:
X, y = get_dataset()
# split dataset into train and test sets
X_train_full, X_test, y_train_full, y_test = ??
print('Train: %s, Test: %s' % (X_train_full.shape, X_test.shape))
models = get_models()
???:
	# ajustamos cada modelo en train
	???
	# hacemos las predicciones
	yhat = ???
	# evaluamos las predicciones
	score = ???
	print('>%s Accuracy: %.3f' % (name, score*100))

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section3"></a> 
# <font color="#004D7F"> 3. Blending para regresión</font>

Veamos ahora para un problema de regresión

<a id="section31"></a> 
## <font color="#004D7F"> 3.1. Resultado Blending</font>

Usamos la función `make_regression()` para crear un problema de regresión sintética con 1000 ejemplos y 20 características de entrada.

In [ ]:
from numpy import mean
from numpy import std
from sklearn.datasets import make_regression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

# get the dataset
def get_dataset():
	X, y = make_regression(n_samples=10000, n_features=20, n_informative=10, noise=0.3, random_state=7)
	return X, y

- Usamos los siguientes algoritmos para modelos de nivel 0: LoR, CART, NB, SVM y k-NN. 
- La función `get_models()` implementa esto y devuelve la lista de modelos a considerar.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`KNeighborsRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`SVR`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html#sklearn.svm.SVR)
</div>

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

def get_models():
	models = list()
	models.append(???)
	models.append(???)
	models.append(???)
	models.append(???)
	return models

La función `fit_ensemble()` utilizada para entrenar Blending no cambia con respecto a la clasificación, excepto que el modelo utilizado para la combinación debe cambiarse a LiR y utilizar MAE.

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression

def fit_ensemble(models, X_train, X_val, y_train, y_val):
	meta_X = list()
	for _, model in models:
		model.fit(X_train, y_train)
		yhat = ???
		yhat = yhat.reshape(len(yhat), 1)
		meta_X.append(yhat)
	meta_X = hstack(meta_X)
	blender = ???
	blender.fit(meta_X, y_val)
	return blender

También tenemos que modificar la función ya que ahora no estamos realizando probabilidades.

In [ ]:
def predict_ensemble(models, blender, X_test):
	# make predictions with base models
	meta_X = list()
	for _, model in models:
		yhat = ???
		yhat = yhat.reshape(len(yhat), 1)
		meta_X.append(yhat)
	meta_X = hstack(meta_X)
	return ???

Vemos los resultados finales

In [ ]:
X, y = get_dataset()
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.5, random_state=1)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.33, random_state=1)
print('Train: %s, Val: %s, Test: %s' % (X_train.shape, X_val.shape, X_test.shape))
models = get_models()
blender = fit_ensemble(models, X_train, X_val, y_train, y_val)
yhat = predict_ensemble(models, blender, X_test)
# evaluamos predicciones
score = ???
print('Blending MAE: %.3f' % score)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: La API de scikit-learn invierte el signo del MAE para transformarlo, de minimizar el error a maximizar el error negativo. Esto significa que los errores positivos de gran magnitud se convierten en grandes errores negativos (por ejemplo, 100 se convierte en -100) y un modelo perfecto no tiene ningún error con un valor de 0,0. También significa que podemos ignorar con seguridad el signo de las puntuaciones MAE medias. 

</div>

<a id="section32"></a> 
## <font color="#004D7F"> 3.2. Modelos contribuyentes</font>

Veamos si mejora en igualdad de condiciones los modelos contribuyentes

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
</div>

In [ ]:
X, y = get_dataset()
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.5, random_state=1)
print('Train: %s, Test: %s' % (X_train_full.shape, X_test.shape))
models = get_models()
for name, model in models:
	???
	yhat = ???
	score = ???
	print('>%s MAE: %.3f' % (name, score))

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: La API de scikit-learn invierte el signo del MAE para transformarlo, de minimizar el error a maximizar el error negativo. Esto significa que los errores positivos de gran magnitud se convierten en grandes errores negativos (por ejemplo, 100 se convierte en -100) y un modelo perfecto no tiene ningún error con un valor de 0,0. También significa que podemos ignorar con seguridad el signo de las puntuaciones MAE medias. 

</div>

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="sectionEj"></a>
<h3><font color="#004D7F" size=6> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicios</font></h3>

Se proponen las siguientes actividades para consolidar el aprendizaje.

# <font color="#004D7F" size=5>Ejercicio 1</font>
__Hiperparámetros__. Explore diferentes configuraciones de hiperparámetros que veas en la librería sobre Blending y comente los resultados. Utilice una estrategia de búsqueda.

# <font color="#004D7F" size=5>Ejercicio 2</font>
__Evaluar algoritmos__. Utilizar diferentes algoritmos a los que hemos usado en esta unidad y evaluar su impacto con Voting usando hiperparáemtros clave.

# <font color="#004D7F" size=5>Ejercicio 3</font>
__Datasets reales__. Busque un dataset original y verdadero (que no sea sintético) y evalúe el uso de los conceptos vistos en esta unidad. Los conjuntos de datos en pueden ser obtenidos del [repositorio de aprendizaje automático de UCI](https://archive.ics.uci.edu/).

# <font color="#004D7F" size=5>Ejercicio 4</font>
__Modelos contribuyente__. Utilice otros modelos contribuyentes diferentes a los vistos y con diferentes hiperparámetros utilizando una estrategia de búsqueda.

# <font color="#004D7F" size=5>Ejercicio 5</font>
__Búsqueda de la mejor configuración__. Como se ha visto existen diferentes hiperparámetros que pueden ajustar nuestro modelo. Haga una búsqueda para un dataset real de cuales, entre un rango amplio de hiperparétros, maximizan la métrica. Puede utilizar una búsqueda aleatoria.

# <font color="#004D7F" size=5>Ejercicio 6</font>
__Incluir más niveles__. En este ejemplo hemos usado el modelo ingresando un solo niel 0 y 1. Se pueden integrar más niveles haciendo una estrategia de integración entre ellos. Se pide integregar más niveles en un dataset real y explicando las limitaciones y fortalezas observadas.

---

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>